In [1]:
import os
from io import StringIO
import numpy as np
import pandas as pd

In [2]:
df_monthly_metrics = pd.read_json("monthly_metrics.jsonl", lines=True)
df_monthly_metrics.shape
# monthly metrics has 1820 rows and 7 columns

(18270, 7)

In [3]:
df_daily_metrics = pd.read_json("daily_metrics.jsonl", lines=True)
df_daily_metrics.shape
#there are 666855 rows and 7 columns in the daily metrics table

(666855, 7)

In [4]:
df_transactions = pd.read_json("transactions.jsonl", lines=True)
df_transactions.shape
# there are 725656 rows and 10 columns in the transactions table

(7254656, 10)

In [5]:
# remove constant & duplicate function
def remove_constant_duplicate_features(df: pd.DataFrame, report: bool = True) -> pd.DataFrame :
    keep = []
    dropped_constants = []
    for col in df.columns :
        if df[col].nunique(dropna = False) <= 1 :
            dropped_constants.append(col)
        else :
            keep.append(col)
    copy = df[keep].copy()

    # drop duplicates
    dropped_values = []
    cols = copy.columns.tolist()
    already_encountered = {}
    for current in cols:
        data = copy[current]
        h = (tuple(data.head(min(len(data), 100)).astype(str)), data.dtype.str, len(data))
        if h in already_encountered and copy[already_encountered[h]].equals(data) :
            dropped_values.append(current)
        else : 
            already_encountered[h] = current
    copy = copy.drop(columns = dropped_values, errors = 'ignore')

    if report:
        if dropped_constants:
            print(f"Dropped constant/no-variance columns: {dropped_constants}")
        if dropped_values:
            print(f"Dropped exact-duplicate columns: {dropped_values}")
        if not dropped_constants and not dropped_values:
            print("No constant or duplicate columns found.")
    
    return copy        


In [6]:
# check correlations function
def check_and_drop_correlated(df: pd.DataFrame, threshold: float = 0.995) -> pd.DataFrame :
    to_drop = set()
    # known pairs 
    pairs = [("cost_sum", "data_sum"), ("cost_mean", "data_mean")]
    for a, b in pairs :
        if a in df.columns and b in df.columns :
            correlation = pd.to_numeric(df[a], errors='coerce').corr(pd.to_numeric(df[b], errors='coerce'))
            print(f"Correlation check: {a} vs {b} = {correlation:.4f}")
            if abs(correlation) >= threshold :
                to_drop.add(b)

            if to_drop :
                print(f"Dropped correlated features: {sorted(to_drop)}")
                df = df.drop(columns = sorted(to_drop), errors = 'ignore')
            return df

In [7]:
# keep best features function
def select_best_features(df: pd.DataFrame) -> pd.DataFrame :
    best_features = [b for b in ["req_count", "cost_sum", "cost_mean"] if b in df.columns]
    print(f"Keeping best features: {best_features}")
    return df[best_features].copy()

In [8]:
# creating new features function
def engineer_new_features(df: pd.DataFrame, daily_metrics: pd.DataFrame = None, lags = (1, 2)) -> pd.DataFrame :
    copy = df.copy()

    # ratios
    if {"cost_sum", "req_count"}.issubset(copy.columns) :
        copy["ratio_cost_per_req"] = copy["cost_sum"] / copy["req_count"].replace({0: np.nan})
    if {"data_sum", "req_count"}.issubset(copy.columns) :
        copy["ratio_data_per_req"] = copy["data_sum"] / copy["req_count"].replace({0, np.nan})

    # log values
    for current in copy.columns :
        if current in ["cost_sum", "cost_mean", "req_count"] :
            copy[f"log1p_{current}"] = np.log1p(copy[current])
        # added log values

    # lag features
    if daily_metrics is not None :
        dm = daily_metrics.copy()
        df["date"] = pd.to_datetime(dm["date"], errors = "coerce")
        dm = dm.sort_values("date")
        metric_cols = [current for current in ["req_count", "cost_sum", "data_sum", "cost_mean"] if current in dm.columns]
        for current in metric_cols :
            for L in lags :
                dm[f"{current}_lag{L}"] = dm[current].shift(L)
        # added lag features
    
    # puts the lag features back in copy if df has 'date'
    if "date" in copy.columns :
        copy = copy.merge(dm[['date'] + [f"{c}_lag{L}" for c in metric_cols for L in lags]], on = 'date', how = 'left')
        # added lag features from daily_metrics

    return copy


In [9]:
# run full pipline function
def run_feature_engineering_pipeline(df: pd.DataFrame, daily_metrics: pd.DataFrame = None) :
    # remove constant and dup features
    cleaned = remove_constant_duplicate_features(df)
    
    # check correlations and dorp redundancy
    cleaned = check_and_drop_correlated(cleaned)

    # keep only best features
    best = select_best_features(cleaned)
    
    # engineering new features
    engineered = engineer_new_features(best, daily_metrics = daily_metrics)

    return cleaned, best, engineered

In [10]:
# pivot monthly metrics
df_monthly_pivot = df_monthly_metrics.pivot_table(
    index=['tenant/id','monthly/created','app/id','app/name'],
    columns='monthly/metric',
    values='monthly/value'
).reset_index()

# renaming monthly metric columns to match pipline labels
df_monthly_pivot.rename(columns={
    'requests_made': 'req_count',
    'cost': 'cost_sum',
    'data_used': 'data_sum',
    'cost_per_request_made': 'cost_mean',  
    'monthly/created': 'date' 
}, inplace=True)

print(f"Monthly pivot shape: {df_monthly_pivot.shape}")

# pivot daily metrics
df_daily_pivot = df_daily_metrics.pivot_table(
    index=['tenant/id','daily/time','app/id','app/name'],
    columns='daily/metric',
    values='daily/value'
).reset_index()

# renaming daily metric columns to match pipline labels
df_daily_pivot.rename(columns={
    'requests_made': 'req_count',
    'cost': 'cost_sum',
    'data_used': 'data_sum',
    'cost_per_request_made': 'cost_mean',
    'daily/time': 'date'  
}, inplace=True)

print(f"Daily pivot shape: {df_daily_pivot.shape}")

# privot transactions
df_transactions_pivot = df_transactions.pivot_table(
    index=['tenant/id','transaction/time','transaction/consumer/id','transaction/consumer/name',
           'transaction/supplier/id','transaction/supplier/name'],
    values=['transaction/data','transaction/cost']
).reset_index()

# rename transaction columns to match pipline labels
df_transactions_pivot.rename(columns={
    'transaction/data': 'data_sum',
    'transaction/cost': 'cost_sum',
    'transaction/time': 'date' 
}, inplace=True)

print(f"Transactions pivot shape: {df_transactions_pivot.shape}")

Monthly pivot shape: (870, 25)
Daily pivot shape: (63510, 25)
Transactions pivot shape: (5186279, 8)


In [11]:
# transactions dataset feature engineering
print("Transactions Feature Engineering Information:")
cleaned_transactions, core_transactions, engineered_transactions = run_feature_engineering_pipeline(
    df_transactions_pivot,
    daily_metrics=df_daily_pivot
)
print()

# daily metrics dataset feature engineering 
print("Daily Metrics Feature Engineering Information:")
cleaned_daily, core_daily, engineered_daily = run_feature_engineering_pipeline(df_daily_pivot)
print()

# monthly metrics dataset feature engineering 
print("Monthly Metrics Feature Engineering Information:")
cleaned_monthly, core_monthly, engineered_monthly = run_feature_engineering_pipeline(df_monthly_pivot)
print()

# Print shapes to confirm
print(f"Transactions: cleaned={cleaned_transactions.shape}, core={core_transactions.shape}, engineered={engineered_transactions.shape}")
print(f"Daily Metrics: cleaned={cleaned_daily.shape}, core={core_daily.shape}, engineered={engineered_daily.shape}")
print(f"Monthly Metrics: cleaned={cleaned_monthly.shape}, core={core_monthly.shape}, engineered={engineered_monthly.shape}")

Transactions Feature Engineering Information:
Dropped constant/no-variance columns: ['tenant/id']
Correlation check: cost_sum vs data_sum = 0.8773
Keeping best features: ['cost_sum']

Daily Metrics Feature Engineering Information:
Dropped constant/no-variance columns: ['tenant/id']
Dropped exact-duplicate columns: ['value']
Correlation check: cost_sum vs data_sum = 0.9718
Keeping best features: ['req_count', 'cost_sum', 'cost_mean']

Monthly Metrics Feature Engineering Information:
Dropped constant/no-variance columns: ['tenant/id', 'outage_cost', 'outage_efficiency', 'outage_impact']
Dropped exact-duplicate columns: ['value']
Correlation check: cost_sum vs data_sum = 0.9719
Keeping best features: ['req_count', 'cost_sum', 'cost_mean']

Transactions: cleaned=(5186279, 7), core=(5186279, 2), engineered=(5186279, 2)
Daily Metrics: cleaned=(63510, 23), core=(63510, 3), engineered=(63510, 7)
Monthly Metrics: cleaned=(870, 20), core=(870, 3), engineered=(870, 7)
